> **History.** Sutskever, Vinyals, and Le (Google, 2014) showed that a fixed-size vector — the final hidden state of an RNN encoder — could represent an entire sentence well enough for machine translation. The bottleneck was the vector size. Bahdanau, Cho, and Bengio (2015) introduced additive attention: instead of one fixed vector, the decoder could read a weighted blend of all encoder hidden states. This was the direct ancestor of the Transformer's cross-attention mechanism. The encoder-decoder pattern still powers T5, BART, mT5, and every sequence-to-sequence production model today.
>
> **Where you are.** You've built a full Transformer in `02-transformers/` — you understand self-attention, positional encoding, residuals, and multi-head attention. The gap: you built decoder-only (MiniLM) and briefly saw the encoder-only variant. You have not yet trained the encoder-decoder architecture — the variant that handles tasks where input and output sequences differ in length and meaning.
>
> **Notation.** $S$ — source sequence length; $T$ — target sequence length; $d_{model}$ — embedding dimension; $Q$ from decoder, $K$/$V$ from encoder in cross-attention; `mask=None` for encoder (bidirectional), causal mask for decoder; teacher forcing: feed ground-truth target at each decoding step during training.


---

## Prerequisite Bridge — From `02-transformers/transformers.ipynb`

| Foundation                                              | Role in this notebook                                                                                                                                |
| ------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------- |
| Multi-head self-attention (Q, K, V, scaled dot-product) | Used directly as `MultiHeadSelfAttention` in both encoder and decoder blocks — no re-derivation                                                      |
| Causal attention mask (triangular)                      | The decoder's causal self-attention uses this exact mask; the encoder deliberately removes it                                                        |
| Sinusoidal positional encoding                          | Carried forward unchanged into the encoder and decoder embeddings                                                                                    |
| Residual connections + LayerNorm (Pre-LN)               | Both blocks follow the same Pre-LN pattern built in `02-transformers`                                                                                |
| Decoder-only architecture (`MiniLM`)                    | This notebook extends that to encoder + cross-attention; the decoder block here is the same decoder-only block plus one new cross-attention sublayer |

> **If you haven't run `02-transformers/transformers.ipynb`** the attention mechanism, causal masking, and Pre-LN architecture used here will not be familiar — those derivations are not repeated. Complete that notebook first.


## 0 · The Challenge

> **The mission**: Build an encoder-decoder Transformer from scratch and prove it learns a non-trivial mapping — integer sequence reversal (`[3, 1, 4, 1] → [1, 4, 1, 3]`).

**What we know so far:**

- Decoder-only Transformers (MiniLM) generate text left-to-right from context.
- The causal mask prevents the decoder from seeing future tokens.
- **But we still can't**: map a variable-length source sequence to a different-length target sequence, or give the decoder access to the full source representation at every decoding step.

**What's blocking us:**
A decoder-only model reads its own previous output. It cannot read a separately encoded source. The bottleneck: a single fixed-size vector (the last encoder hidden state) loses positional specificity — cosine similarity between encoder outputs collapses toward the mean.

**What this chapter unlocks:**
Cross-attention — the decoder queries the encoder's full output at every step. Every source token's representation is available to every decoding step. The anti-diagonal pattern in the trained cross-attention map will prove the model learned the reversal.


## Encoder-Decoder Transformers in PyTorch

## From Sequence Reversal to seq2seq Translation

This notebook builds an encoder-decoder transformer from first principles in PyTorch, using a single running example — reversing a short sequence of integers — to make every moving part observable before bridging to T5/BART.

| Step | Part                  | Concept                                   | Key Idea                                                      |
| ---- | --------------------- | ----------------------------------------- | ------------------------------------------------------------- |
| 1    | The Contract          | What encoder-decoder solves               | Variable-length I/O + bidirectionality                        |
| 2    | The Encoder           | Bidirectional attention                   | Every token sees every other token                            |
| 3    | The Bottleneck        | Why naive concat fails                    | Fixed vector cannot hold a long sequence                      |
| 4    | Cross-Attention       | The bridge                                | Q = decoder, K = V = encoder                                  |
| 5    | Full Model + Training | Wire encoder + cross-attn + decoder       | Train on sequence reversal                                    |
| 6    | Cross-Attention Map   | Visualise what was learned                | Decoder step $i$ attends to source position $n-i$             |
| 6a   | Free-Running Decoding | Teacher forcing vs. autoregressive greedy | Exposure bias measured directly (+ a short beam-search aside) |
| 7    | Toy to Real           | T5 / BART parameter mapping               | Same architecture, wider vectors                              |


## The Encoder-Decoder Contract at a Glance

![Encoder-decoder sequence-to-sequence contract: bidirectional source encoding, cross-attention, and causal target generation](images/encoder-decoder-contract.png)

The encoder produces a contextual representation for each source token. The decoder generates the target sequence one step at a time, using causal self-attention over earlier target tokens and cross-attention over the complete encoder output.


The full encode-decode pipeline at inference time:

```mermaid
graph LR
    Src[Source tokens] --> EmbEnc[Embed + PE]
    EmbEnc --> Enc["Encoder\n(N × bidirectional SA → FFN)"]
    Enc --> Ctx["Context vectors\n(one per source token)"]

    Tgt[Target tokens so far] --> EmbDec[Embed + PE]
    EmbDec --> DecSA["Decoder\n(causal SA)"]
    DecSA --> CrossAttn["Cross-Attention\nQ ← decoder, K/V ← Context"]
    Ctx --> CrossAttn
    CrossAttn --> FFN[FFN → lm_head]
    FFN --> Out[Next target token]
```

Key: the encoder runs **once** over the full source; the decoder runs **step-by-step**, querying the fixed context vectors at each position via cross-attention.


In [ ]:
# ── Setup: imports and reproducibility seed ───────────────────────────────────
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Seed fixed at 42 — every cell in this notebook is deterministic.")
print("  -> Re-running any cell produces the same numbers as in the prose above it.")

---

## Part 1 — The Encoder-Decoder Contract

### What problem does it solve that decoder-only cannot?

A decoder-only model (GPT-style) produces one token at a time, conditioned on every
token that came before in a single flat sequence. It is excellent at language modelling
— but consider **sequence reversal**: input `[3, 1, 4, 1]` must produce `[1, 4, 1, 3]`.
The first output token (`1`) depends on the _last_ input token (`1` at position 3).
A causal decoder, at generation step 0, cannot look forward to see position 3.

The 2x2 taxonomy of sequence-to-sequence tasks:

|                             | Same vocabulary / same length   | Variable-length output                              |
| --------------------------- | ------------------------------- | --------------------------------------------------- |
| **Unidirectional (causal)** | Language modelling (GPT, LLaMA) | Hard: future source tokens unavailable              |
| **Bidirectional**           | Sentence classification (BERT)  | **Encoder-Decoder**: T5, BART, original Transformer |

**The encoder-decoder contract:**

- The **encoder** reads the full source sequence in both directions and produces one
  enriched context vector per source token. It does not generate; it _enriches_.
- The **decoder** generates the target sequence autoregressively, querying the full
  source map via cross-attention at every step.

The cross-attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

where $Q$ comes from the **decoder** state and $K, V$ come from the **encoder** output.
The $\sqrt{d_k}$ scaling prevents dot-products from growing so large that softmax
saturates — a problem we will measure in Part 2.


In [ ]:
# ── Running example: integer sequence reversal ────────────────────────────────
#
# Toy vocabulary: digits 0-9 plus three special tokens.
# Task: reverse a length-4 sequence, e.g. [3, 1, 4, 1] -> [1, 4, 1, 3].
# This forces cross-attention to learn a non-trivial src->tgt routing:
#   output position 0 must attend to source position 3, etc.

PAD, BOS, EOS = 10, 11, 12
VOCAB_SIZE = 13  # 0-9 digits + PAD + BOS + EOS
SEQ_LEN = 4  # source / target sequence length
D_MODEL = 32  # embedding / hidden dimension (tiny for visibility)
N_HEADS = 4
D_FF = 64
N_LAYERS = 2


def make_reversal_pairs(n_samples, seq_len=SEQ_LEN, seed=42):
    """Generate (src, tgt) pairs where tgt = list(reversed(src))."""
    rng = np.random.default_rng(seed)
    srcs, tgts = [], []
    for _ in range(n_samples):
        src = list(rng.integers(0, 10, size=seq_len))
        tgt = src[::-1]
        srcs.append(src)
        tgts.append(tgt)
    return srcs, tgts


train_srcs, train_tgts = make_reversal_pairs(2000)
val_srcs, val_tgts = make_reversal_pairs(200, seed=99)

print("Reversal task examples (first 5):")
print(f"  {'Source':<20}  Target (reversed)")
print(f"  {'-'*18}  {'-'*18}")
for s, t in zip(train_srcs[:5], train_tgts[:5]):
    print(f"  {str(s):<20}  {str(t)}")
print()
print(f"Training pairs : {len(train_srcs)}")
print(f"Vocab size     : {VOCAB_SIZE}  (0-9 = digits, 10=PAD, 11=BOS, 12=EOS)")
print("Decoder input  : [BOS] + target[:-1]  (teacher forcing)")
print("Decoder target : target + [EOS]")

---

## Part 2 — The Encoder: Enriching Source Representations

### Bidirectional self-attention

In a **decoder** block, position $i$ can attend only to positions $0, \ldots, i$.
This is enforced by adding $-\infty$ to the upper-triangle of the score matrix before
the softmax. In an **encoder** block, we simply omit that mask. Every token sees
every other token — backward _and_ forward — in the very first block.

The complete implementation difference between an encoder and a decoder block is one
argument:

```
decoder block:  self_attn(x, mask=causal_mask)  # upper-triangle -> -inf
encoder block:  self_attn(x, mask=None)          # nothing blocked
```

Everything else — `MultiHeadSelfAttention`, `LayerNorm`, `FeedForward` — is identical.
We will confirm this with the heatmap experiment below.


### How the Encoder Builds Source Representations

![Encoder pipeline from source embeddings and positional information through bidirectional self-attention to one contextual representation per source token](images/encoder-source-representations.png)

The encoder produces one contextual vector for every source position, rather than a single fixed summary. Its self-attention is bidirectional, so each source token can use information from every other source token.


In [ ]:
# ── MultiHeadSelfAttention: the shared attention primitive ────────────────────
#
# Variable names mirror the math:
#   W_Q, W_K, W_V  projection matrices
#   Q, K, V        projected queries, keys, values
#   d_k            per-head dimension  (d_model // n_heads)
# The optional `mask` argument is the ONLY thing that will later distinguish
# an encoder block (mask=None) from a decoder block (mask=causal_mask).


class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention with optional causal mask."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        # Project and reshape to (B, n_heads, S, d_k)
        Q = self.W_Q(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        # Scaled dot-product: softmax( Q K^T / sqrt(d_k) ) V
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # mask: (S, S) bool; True = blocked position
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        attn_w = F.softmax(scores, dim=-1)  # (B, n_heads, S, S)
        out = torch.matmul(attn_w, V)  # (B, n_heads, S, d_k)
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out), attn_w

Every attention sublayer is paired with a position-wise feed-forward network — a
small 2-layer MLP applied independently to each token's vector, giving the model
nonlinear capacity that attention (a weighted average) cannot provide alone.


In [ ]:
# ── FeedForward: position-wise sublayer applied after every attention block ──
class FeedForward(nn.Module):
    """Position-wise FFN: Linear(d_model -> d_ff) + ReLU + Linear(d_ff -> d_model)."""

    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

Combine the attention primitive above with a feed-forward sublayer and residual
connections into one `EncoderBlock`. The `mask=None` passed to `self.attn` is the
single line that makes this block bidirectional rather than causal.


In [ ]:
# ── EncoderBlock: attention + FFN with residual connections ──────────────────
class EncoderBlock(nn.Module):
    """Encoder layer: bidirectional self-attn (mask=None) + FFN, both with residual."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        # Pre-norm residual connections (GPT-2 / T5 style)
        attn_out, attn_w = self.attn(
            self.norm1(x), mask=None
        )  # <- mask=None is the key
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_w

Encoder blocks alone don't know token order — attention treats a sequence as an
unordered set of vectors. Add fixed sinusoidal position encodings to the token
embeddings, then stack `n_layers` `EncoderBlock`s into the full `MiniEncoder`.


In [ ]:
# ── sinusoidal_pe + MiniEncoder: position encodings and the full stack ────────
def sinusoidal_pe(max_seq, d_model):
    """Return sinusoidal positional encoding of shape (max_seq, d_model)."""
    pe = torch.zeros(max_seq, d_model)
    pos = torch.arange(0, max_seq, dtype=torch.float).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class MiniEncoder(nn.Module):
    """
    Bidirectional encoder (BERT / T5-encoder style).
    Output: one enriched d_model-dimensional vector per source token.
    NOT a next-token predictor — a context enricher.
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.register_buffer("pe", sinusoidal_pe(max_seq, d_model))
        self.blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.norm_out = nn.LayerNorm(d_model)

    def forward(self, token_ids):
        S = token_ids.shape[1]
        x = self.token_emb(token_ids) + self.pe[:S]
        all_attn = []
        for block in self.blocks:
            x, aw = block(x)
            all_attn.append(aw)
        return self.norm_out(x), all_attn

With `MiniEncoder` assembled, sanity-check its output shapes on our running example
`[3, 1, 4, 1]` before trusting it anywhere else in this notebook.


In [ ]:
# ── Sanity check ───────────────────────────────────────────────────────────────
torch.manual_seed(42)
enc_test = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
dummy_ids = torch.tensor([[3, 1, 4, 1]])
enc_out, enc_attns = enc_test(dummy_ids)

print("Encoder output shape:", tuple(enc_out.shape))
print(f"  -> (batch=1, seq_len={SEQ_LEN}, d_model={D_MODEL})")
print("  -> One enriched vector per source token, NOT a next-token prediction")
print()
print("Attention weight shape per block:", tuple(enc_attns[0].shape))
print(f"  -> (batch=1, n_heads={N_HEADS}, src={SEQ_LEN}, src={SEQ_LEN})")
print("  -> Every source token can attend to every other source token (no mask)")

### Code Walkthrough: MiniEncoder Building Blocks, Recapped

**The pieces above, split into small cells — 4 key patterns worth re-emphasizing:**

---

**`W_Q(x).view(B, S, n_heads, d_k).transpose(1, 2)` — split heads for parallel attention**
Each token's `d_model`-dimensional embedding is projected to Q/K/V then split into `n_heads` independent sub-vectors of size `d_k = d_model // n_heads`. The `.transpose(1, 2)` swaps sequence and head axes, giving shape `(B, n_heads, S, d_k)`. Each head independently attends over the full sequence but works in its own lower-dimensional subspace.

---

**`mask.unsqueeze(0).unsqueeze(0)` — broadcast the mask over batch and head dims**
The raw mask has shape `(S, S)`. Two `.unsqueeze(0)` calls prepend a batch dim and a head dim, yielding `(1, 1, S, S)`. PyTorch broadcasts this over all batches and all heads without allocating extra memory — the same pattern appears in every production GPT/BERT implementation. Passing `mask=None` in `EncoderBlock` skips this entirely, giving bidirectional (unrestricted) attention.

---

**`sinusoidal_pe(max_seq, d_model)` — fixed positional fingerprints added to embeddings**
Even-indexed dimensions use $\sin(pos / 10000^{2i/d})$ and odd dimensions use $\cos(\ldots)$. Different-frequency sinusoids produce a unique fingerprint for every position that never repeats. Adding PE to token embeddings lets the model see _both_ what a token is and where it sits — without any trainable parameters.

---

**`EncoderBlock` with `mask=None` — bidirectional self-attention is the encoder's superpower**
`self.attn(self.norm1(x), mask=None)` removes the causal restriction. Token at position 0 directly attends to position 99 in a single layer, and vice versa. This is the key difference from a decoder: the encoder builds a rich **context map** for every source position, not a next-token predictor.

> **PyTorch shape note:** The encoder output is `(B, seq_len, d_model)` — one enriched vector _per source token_, not a single sequence vector. Cross-attention later queries these per-position vectors individually, which is why the decoder can attend to any source position at each generation step.


#### Predict before you run — encoder heatmap row 0

In the decoder's causal heatmap, row 0 has exactly **1** non-zero cell (token 0
attends only to itself — it cannot see the future).

**Predict:** In the encoder heatmap, how many non-zero cells will row 0 have?

A. 1 (same as decoder — only itself)
B. 2 (attends to immediate neighbours only)
C. 4 (attends to all positions equally)

Write your answer, then run the encoder-vs-decoder heatmap experiment below to see the reveal.


In [ ]:
# ── Encoder vs Decoder attention heatmap: SAME weights, ONLY the mask differs ──
#
# By running the same MultiHeadSelfAttention layer twice — once with mask=None
# and once with a causal mask — we isolate the mask as the single causal variable.

torch.manual_seed(7)
shared_mha = MultiHeadSelfAttention(D_MODEL, N_HEADS)

x_demo = torch.randn(1, SEQ_LEN, D_MODEL)  # same input for both runs

# Upper-triangle mask: True = blocked (sent to -inf before softmax)
causal_mask = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool), diagonal=1)

_, w_encoder = shared_mha(x_demo, mask=None)  # bidirectional
_, w_decoder = shared_mha(x_demo, mask=causal_mask)  # causal

# Head 0 for display
w_enc_h0 = w_encoder[0, 0].detach().numpy()
w_dec_h0 = w_decoder[0, 0].detach().numpy()
labels = ["3", "1", "4", "1"]  # our running example token values

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, title, cmap in [
    (axes[0], w_enc_h0, "Encoder (mask=None)\nBidirectional", "Blues"),
    (axes[1], w_dec_h0, "Decoder (causal mask)\nLower-triangle only", "Oranges"),
]:
    # cbar=True (the default): a continuous-magnitude heatmap needs its own
    # colorbar as the legend for what a color means — annot=True alone is not
    # a substitute once the reader compares two differently-colored panels.
    sns.heatmap(
        data,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        xticklabels=labels,
        yticklabels=labels,
        linewidths=0.5,
        cbar=True,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "attention weight"},
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    ax.tick_params(axis="x", rotation=0)

plt.suptitle(
    "Same MHA weights, same input — only the mask differs",
    fontsize=11,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

nz_enc = int((w_enc_h0[0] > 0.01).sum())
nz_dec = int((w_dec_h0[0] > 0.01).sum())
print(f"Row 0 non-zero cells — Encoder: {nz_enc}   Decoder: {nz_dec}")
print(f"  -> Encoder row 0 attends to ALL {SEQ_LEN} positions  (answer: C)")
print(f"  -> Decoder row 0 attends to only 1 position (cannot see the future)")
print()
print("Implementation difference: one argument — mask=None vs mask=causal_mask.")
print("  -> Every other component (MHA, FFN, LayerNorm) is shared code.")

#### What just happened — and what is missing

We proved that `mask=None` gives every token a 360-degree view of the source.
Token `4` at position 2 can incorporate signals from `3` (position 0) and `1`
(position 3) in the very first block.

We now have a box of enriched source vectors. The question is: how does the decoder
consume them? The naive approach — concatenate encoder output to the decoder context —
has a hidden flaw. Part 3 exposes it.


#### Your turn — how many heads make attention patterns diverge?

Part 2 built `MultiHeadSelfAttention` with `N_HEADS=4`. Each head learns its own
`W_Q`/`W_K`/`W_V` and therefore its own attention pattern over the same input.

**Predict:** with only **1** head, is there anything to compare (a single pattern is
trivially identical to itself)? With **4** or **8** heads, will the patterns be nearly
identical (redundant) or clearly different (specialised)?

Change `ex_n_heads` below and re-run to see how head count changes the diversity of
attention patterns computed on our running example `[3, 1, 4, 1]`.


### Why Cross-Attention Replaces a Fixed Context Vector

![Historical fixed-context bottleneck compared with a Transformer encoder-decoder using cross-attention over all encoder outputs](images/fixed-context-bottleneck.png)

Earlier sequence-to-sequence models often compressed the source into one fixed-size vector. Cross-attention lets each decoding step consult the full set of encoder representations instead, so different target positions can focus on different source tokens.


In [ ]:
# ── EXERCISE — head-count vs. attention-pattern diversity ────────────────────
# Change ex_n_heads (must divide D_MODEL=32: try 1, 2, 4, 8) and predict whether
# more heads produce more DIFFERENT attention patterns on our running example.

ex_n_heads = 4  # CHANGE: try 1, then 2, then 8

torch.manual_seed(42)
mha_ex = MultiHeadSelfAttention(D_MODEL, ex_n_heads)
x_ex = torch.randn(1, SEQ_LEN, D_MODEL)
_, w_ex = mha_ex(x_ex, mask=None)  # bidirectional, same as MiniEncoder

patterns = w_ex[0].detach().reshape(ex_n_heads, -1).numpy()
print(f"{ex_n_heads} head(s) on our running example [3, 1, 4, 1]:")

if ex_n_heads > 1:
    corrs = np.corrcoef(patterns)
    off_diag = corrs[~np.eye(ex_n_heads, dtype=bool)]
    print(
        f"  Pairwise correlation between head attention patterns: mean={off_diag.mean():.3f}"
    )
    if off_diag.mean() < 0.5:
        print("  -> Heads show CLEARLY DIFFERENT attention patterns (specialised).")
    else:
        print("  -> Heads are still fairly correlated at this random init (untrained).")
else:
    print(
        "  -> Only 1 head exists — nothing to compare against, correlation undefined."
    )

print(
    f"  -> Each head works in a {D_MODEL // ex_n_heads}-dim subspace "
    f"(d_k = d_model / n_heads = {D_MODEL}/{ex_n_heads})."
)

---

## Part 3 — The Bottleneck Problem

### Why passing encoder output as decoder initial state fails

Pre-attention seq2seq models (Sutskever et al., 2014) compressed the entire source
sequence into a single fixed-size vector — the last hidden state of an RNN encoder —
and handed it to the decoder as its initial hidden state. This is the **information
bottleneck**.

The capacity of a $d$-dimensional vector is fixed regardless of source length:

$$\text{bottleneck capacity} \propto d_{\text{model}}$$

For a 3-word sentence that capacity might suffice. For a 50-word paragraph it does not —
the vector must carry everything the decoder will ever need, and accuracy collapses on
anything beyond ~30 words. This was a fundamental limitation of early seq2seq systems.

Cross-attention eliminates the bottleneck by keeping **all** $S$ source vectors
simultaneously accessible:

$$\text{cross-attention capacity} \propto d_{\text{model}} \times S_{\text{src}}$$

For $S_{\text{src}} = 50$ tokens that is $50\times$ the information access of the
single-vector approach.


#### Predict before you run — does mean-pooling preserve positional identity?

We are about to measure cosine similarity between per-position encoder vectors and
mean-pooled encoder vectors for two very different source sequences:
`[3, 1, 4, 1]` vs `[9, 8, 7, 6]`.

**Predict:** After mean-pooling the encoder output, will the two sequences be:

A. Clearly distinct (cosine similarity < 0.3) — mean pool retains full positional info
B. Moderately similar (0.3 to 0.7) — some information lost
C. Very similar (> 0.7) — pooling collapses positional differences

Write your answer, then run the cosine-similarity experiment below to find out.


In [ ]:
# ── Prove the bottleneck: mean-pooled encoder vector loses positional detail ───
#
# Strategy: compare two different source sequences.
# Per-position encoder outputs should be clearly distinct (different content).
# Mean-pooled encoder output collapses them toward each other.
#
# We measure cosine similarity between seq_a and seq_b at each encoder position
# and compare to the mean-pooled cosine similarity.

torch.manual_seed(42)
enc_probe = MiniEncoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
enc_probe.eval()

seq_a = torch.tensor([[3, 1, 4, 1]])  # our main example
seq_b = torch.tensor([[9, 8, 7, 6]])  # completely different sequence

with torch.no_grad():
    out_a, _ = enc_probe(seq_a)  # (1, 4, 32)
    out_b, _ = enc_probe(seq_b)  # (1, 4, 32)


def cos_sim(a, b):
    a = a / (a.norm() + 1e-9)
    b = b / (b.norm() + 1e-9)
    return float((a * b).sum())


pos_sims = [cos_sim(out_a[0, i], out_b[0, i]) for i in range(SEQ_LEN)]
pool_sim = cos_sim(out_a.mean(dim=1)[0], out_b.mean(dim=1)[0])

print("Cosine similarity: seq_a=[3,1,4,1] vs seq_b=[9,8,7,6]")
print()
print("Per-position encoder vectors:")
for i, s in enumerate(pos_sims):
    bar = "#" * int(abs(s) * 20)
    tag = "distinct" if abs(s) < 0.7 else "similar"
    print(f"  position {i}: {s:+.4f}  {bar}  ({tag})")
print()
print(f"Mean-pooled vector: {pool_sim:+.4f}")
print()
print("  -> Individual encoder positions encode content distinctly.")
print("  -> Mean-pooling dilutes positional specificity.")
print(f"  -> With longer sequences (S=50) the mean pool becomes a 'blur'.")
print()
print("Cross-attention keeps all", SEQ_LEN, "source vectors alive.")
print(
    "  -> The decoder queries exactly the positions it needs at each generation step."
)

#### What just happened — and what is missing

We measured that mean-pooling the encoder output loses per-position distinction.
The fix is to keep all $S$ source vectors available — and let the decoder dynamically
choose which ones to attend to at each generation step. That dynamic query mechanism
is cross-attention. Part 4 builds it.


---

## Part 4 — Cross-Attention: The Bridge

### Two problems with the naive alternatives

Part 3 proved one naive alternative — compressing the source into a single fixed
vector — loses positional detail. There is a second naive alternative worth ruling
out before introducing cross-attention: _why not just feed the encoder's per-position
vectors into the decoder as extra tokens in its own self-attention sequence?_

**Problem 1 — The bottleneck (recap from Part 3).** A single pooled vector cannot
carry $S$ positions' worth of distinct content once $S$ grows past a handful of
tokens — we measured this directly with cosine similarity above.

**Problem 2 — Concatenation into self-attention breaks on two counts.** Even if we
tried to sidestep the bottleneck by prepending the encoder's $S$ enriched vectors to
the decoder's own sequence and running one shared self-attention over all of it:

- The decoder's **causal mask still applies** to every position in that combined
  sequence — a source token sitting at position 5 would be invisible to a decoder
  token at position 2, even though the whole point of the encoder is that the
  decoder should see _all_ source positions regardless of its own generation step.
- **Source length and target length don't move together.** Our reversal task fixes
  $S_{\text{src}} = 4$, but a real translation task has $S_{\text{src}} \ne T_{\text{tgt}}$
  in general, and $T$ grows by one every generation step while $S$ never changes. A
  single shared self-attention block would need the combined sequence length to be
  re-declared every step — there is no fixed $(S{+}T) \times (S{+}T)$ matrix to mask.

Cross-attention solves both at once by keeping $Q$ and $K/V$ as two genuinely
separate streams, computed once (the encoder side) and re-queried at every step
(the decoder side):

$$Q = \text{decoder state} \cdot W_Q \qquad K = \text{encoder output} \cdot W_K \qquad V = \text{encoder output} \cdot W_V$$

The decoder asks: _"given what I have generated so far ($Q$), which part of the
source text ($K, V$) do I need next?"_

### Correcting a common shorthand: "decoder attention is always causal"

A common mental shortcut is that every attention computation inside a decoder block
must be causally masked, because "decoders generate autoregressively." That shortcut
is only true for the decoder's **self**-attention sub-layer. Cross-attention has
**no mask at all** on the encoder side — not a relaxed mask, none whatsoever —
because $Q$ and $K/V$ come from two different sequences that were never generated
in a shared order in the first place. There is no "future" to hide from a query
that isn't itself part of the sequence being masked.

Because the encoder output is computed once and held fixed, the decoder re-queries
the full source map at every generation step with zero re-computation cost.

Notice the **asymmetry**: the score matrix is $(T_{\text{tgt}} \times S_{\text{src}})$,
not the $(S \times S)$ of self-attention. No mask is applied to the encoder dimension —
the decoder is free to attend to **any** source position regardless of generation step.

```
Decoder self-attn   Q, K, V from decoder state   (causal mask applied)
         |
Cross-attention     Q from decoder, K/V from encoder   (no mask on encoder side)
         |
FFN                 per-token nonlinear transformation
```


### Cross-Attention: Decoder Queries, Encoder Keys and Values

![Cross-attention bridge where decoder hidden states create queries and encoder outputs create keys and values](images/cross-attention-bridge.png)

Cross-attention forms queries from decoder hidden states and keys and values from encoder outputs. Each target position receives a weighted mixture of source information, and the target length and source length may differ.


In [ ]:
# ── CrossAttention module: Q from decoder, K/V from encoder ──────────────────
#
# The decoder's Q vector asks: "what do I need from the source?"
# The encoder's K/V matrices answer: "here is what each source position contains."
# No mask is applied on the encoder (src) dimension.


class CrossAttention(nn.Module):
    """
    Cross-attention layer used inside each decoder block.

    forward(decoder_x, encoder_kv) ->  output (B, T, d_model),
                                        attn_weights (B, n_heads, T, S)
    """

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # projects decoder
        self.W_K = nn.Linear(d_model, d_model, bias=False)  # projects encoder
        self.W_V = nn.Linear(d_model, d_model, bias=False)  # projects encoder
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, decoder_x, encoder_kv):
        B, T, _ = decoder_x.shape
        S = encoder_kv.shape[1]

        Q = self.W_Q(decoder_x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(encoder_kv).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(encoder_kv).view(B, S, self.n_heads, self.d_k).transpose(1, 2)

        # Score matrix: (B, n_heads, T, S)  — asymmetric T x S
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_w = F.softmax(scores, dim=-1)  # (B, n_heads, T, S)

        out = torch.matmul(attn_w, V)  # (B, n_heads, T, d_k)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_O(out), attn_w

With the class defined, run a small demo: one decoder query step attending over the
4 encoder positions for our running example, so the shapes and the attention
distribution are visible before `CrossAttention` gets wired into a full decoder block.


In [ ]:
# ── Demo: 1 decoder query attending to 4 encoder positions ────────────────────
torch.manual_seed(42)
ca_demo = CrossAttention(D_MODEL, N_HEADS)
enc_dummy = torch.randn(1, SEQ_LEN, D_MODEL)  # encoder output for 4 src tokens
dec_dummy = torch.randn(1, 1, D_MODEL)  # one decoder step

ca_out, ca_w = ca_demo(dec_dummy, enc_dummy)

print("Cross-attention shapes:")
print(f"  Decoder Q   : {tuple(dec_dummy.shape)}  (1 decoder token)")
print(f"  Encoder K/V : {tuple(enc_dummy.shape)}  ({SEQ_LEN} source tokens)")
print(f"  Score matrix: {tuple(ca_w.shape)}")
print(f"  Output      : {tuple(ca_out.shape)}")
print()

# Show where attention mass lands (head 0, query 0)
w_h0 = ca_w[0, 0, 0].detach().numpy()
print("Attention weights over source positions (head 0, 1 query step):")
for pos, w in enumerate(w_h0):
    bar = "#" * int(w * 30)
    print(f"  src[{pos}]: {w:.3f}  {bar}")
print()
print("  -> The single decoder query scored ALL source positions.")
print("  -> Which source position gets the highest weight is learned from the")
print("     downstream prediction loss, not hard-coded.")

### Code Walkthrough: CrossAttention Module, Recapped

**The class and its demo above, split into two cells — 3 key patterns worth re-emphasizing:**

---

**`W_Q` projects decoder; `W_K` / `W_V` project encoder — two input streams**
The three projection matrices draw from different sources: `W_Q(decoder_x)` converts the decoder's current state into a query ("what do I need right now?"), while `W_K(encoder_kv)` and `W_V(encoder_kv)` convert all encoder positions into keys and values ("what does each source position contain?"). This is the exact architectural moment where encoder and decoder communicate.

---

**Score matrix `(B, n_heads, T, S)` — asymmetric by design**
`torch.matmul(Q, K.transpose(-2, -1))` produces a `(T, S)` matrix per head — not the `(S, S)` square of self-attention. `T` is the current decoder sequence length; `S` is the fixed source length. As the decoder generates more tokens, T grows; S never changes because the encoder output is computed once and held frozen.

```python
scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
# Q: (B, n_heads, T, d_k)    K^T: (B, n_heads, d_k, S)
# scores: (B, n_heads, T, S)  ← T x S, NOT S x S
```

---

**Demo bar chart — attention weights before training are roughly uniform**
The printed bar chart shows one decoder step distributing probability mass across all 4 source positions. Before training this is near-uniform (random projection). After 30 epochs of reversal training, step 0 should concentrate mass on source position 3 (the last source token), step 1 on position 2, etc. — the anti-diagonal pattern confirmed by the heatmap further down this notebook (Part 6).


#### Your turn — does cross-attention stay well-defined when T ≠ S?

The demo above ran exactly 1 decoder step against `SEQ_LEN=4` source positions.
Part 4 noted the cross-attention score matrix is asymmetric:
$(T_{\text{tgt}} \times S_{\text{src}})$, not $(S \times S)$.

**Predict:** if the decoder has **6** steps but the source still has 4 positions,
what shape will the cross-attention score matrix be — `(6, 4)`, `(4, 6)`, or `(6, 6)`?

Change `ex_t_steps` below and re-run to check.


In [ ]:
# ── EXERCISE — cross-attention shape when T != S ──────────────────────────────
# Change ex_t_steps (decoder length) independent of the source length (SEQ_LEN=4)
# and predict the resulting score-matrix shape BEFORE running.

ex_t_steps = 6  # CHANGE: try 1, 4, 6, 10 — source length stays fixed at SEQ_LEN

torch.manual_seed(42)
ca_ex = CrossAttention(D_MODEL, N_HEADS)
enc_ex = torch.randn(1, SEQ_LEN, D_MODEL)  # source: SEQ_LEN positions, fixed
dec_ex = torch.randn(1, ex_t_steps, D_MODEL)  # target: ex_t_steps positions, varies

out_ex, w_ex = ca_ex(dec_ex, enc_ex)

print(f"Decoder steps (T) = {ex_t_steps},  Source positions (S) = {SEQ_LEN}")
print(
    f"  Score matrix shape : {tuple(w_ex.shape)}  -> (batch, n_heads, T={ex_t_steps}, S={SEQ_LEN})"
)
print(
    f"  Output shape       : {tuple(out_ex.shape)}  -> (batch, T={ex_t_steps}, d_model)"
)
print()
if w_ex.shape[-2] == ex_t_steps and w_ex.shape[-1] == SEQ_LEN:
    print(
        f"  -> Confirmed: cross-attention is asymmetric ({ex_t_steps} x {SEQ_LEN}), never (S x S)."
    )
    print(
        "  -> The decoder can take ANY number of steps; the source length never changes shape."
    )

#### Predict before you run — what will the trained cross-attention map look like?

After training on the reversal task, the cross-attention map should show a
systematic pattern.

For the sequence `[3, 1, 4, 1]` -> `[1, 4, 1, 3]`:

- Decoder step 0 must output `1` — which is at **source position 3**
- Decoder step 1 must output `4` — which is at **source position 2**
- Decoder step 2 must output `1` — which is at **source position 1**
- Decoder step 3 must output `3` — which is at **source position 0**

**Predict:** the trained cross-attention map will look like:

A. The identity matrix (high weight on the diagonal: step 0 attends to src 0, etc.)
B. The anti-diagonal (step 0 attends to src 3, step 1 to src 2, etc.)
C. Uniform attention (each step spreads weight equally over all source positions)

Write your answer. Part 6 reveals it.


---

## Part 5 — Full Encoder-Decoder: Training

### Wiring encoder + cross-attention + decoder

The complete model stacks three components:

1. **Encoder** — bidirectional blocks; produces source map $(B, S, D)$
2. **Decoder** — causal self-attention + cross-attention at every block; generates
   target logits $(B, T, \text{vocab})$
3. **Language model head** — linear projection from $D$ to vocabulary size

The decoder input during training is the **teacher-forced** target:
`[BOS] + target[:-1]`. The decoder output is shifted: `target + [EOS]`.
The loss is cross-entropy averaged over all target positions.


In [ ]:
# ── DecoderBlock (causal self-attn + cross-attn + FFN) ────────────────────────
#
# Three sub-layers following Vaswani et al. (2017):
#   1. Causal self-attention  — decoder reads its own past generated tokens
#   2. Cross-attention        — decoder queries the encoder source map
#   3. Feed-forward           — per-position nonlinear transformation
# Each sub-layer uses a pre-norm residual connection.


class DecoderBlock(nn.Module):
    """Decoder layer: causal self-attn + cross-attn + FFN."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, encoder_out, causal_mask=None):
        sa_out, sa_w = self.self_attn(self.norm1(x), mask=causal_mask)
        x = x + sa_out
        ca_out, ca_w = self.cross_attn(self.norm2(x), encoder_out)
        x = x + ca_out
        x = x + self.ffn(self.norm3(x))
        return x, sa_w, ca_w

With `MiniEncoder` and `DecoderBlock` both defined, wire them into one `EncoderDecoder` module:
encode the source once, then run the `DecoderBlock` stack over the teacher-forced target,
projecting the final hidden state to vocabulary logits with `lm_head`.


In [ ]:
# ── EncoderDecoder: wires MiniEncoder + a DecoderBlock stack + an LM head ─────
#
# encode(src_ids)   runs MiniEncoder ONCE per source sequence.
# decode(...)       runs the DecoderBlock stack over the (teacher-forced) target,
#                   re-using the same encoder_out at every generation step.
# forward(...)      builds the causal mask, then calls encode() then decode() —
#                   this is what every training/eval cell below actually calls.


class EncoderDecoder(nn.Module):
    """
    Full encoder-decoder transformer: a bidirectional MiniEncoder + a stack of
    DecoderBlocks (causal self-attn + cross-attn + FFN) + a linear LM head.
    forward(src_ids, tgt_ids) -> logits (B, T, vocab_size), ca_w (B, n_heads, T, S)
    ca_w is the cross-attention weight from the LAST decoder block only.
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.encoder = MiniEncoder(
            vocab_size, d_model, n_heads, d_ff, n_layers, max_seq
        )
        self.dec_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.register_buffer("dec_pe", sinusoidal_pe(max_seq, d_model))
        self.dec_blocks = nn.ModuleList(
            [DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.dec_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def encode(self, src_ids):
        """Run the bidirectional encoder ONCE; its output is reused by every decoder step."""
        encoder_out, _ = self.encoder(src_ids)
        return encoder_out

    def decode(self, tgt_ids, encoder_out, causal_mask):
        """Run the causal self-attn + cross-attn + FFN stack over tgt_ids."""
        T = tgt_ids.shape[1]
        x = self.dec_emb(tgt_ids) + self.dec_pe[:T]
        ca_w = None
        for block in self.dec_blocks:
            x, _, ca_w = block(x, encoder_out, causal_mask=causal_mask)
        x = self.dec_norm(x)
        return self.lm_head(x), ca_w

    def forward(self, src_ids, tgt_ids):
        encoder_out = self.encode(src_ids)
        T = tgt_ids.shape[1]
        # diagonal=1 leaves (i, i) unmasked; only strictly-future positions (j > i) are blocked
        causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        return self.decode(tgt_ids, encoder_out, causal_mask)

Instantiate the full `EncoderDecoder`, count its parameters, and run one forward
pass on our running example to confirm every shape lines up before training it.


In [ ]:
# ── Architecture inspection ────────────────────────────────────────────────────
torch.manual_seed(42)
model = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
n_params = sum(p.numel() for p in model.parameters())
print(
    f"EncoderDecoder: vocab={VOCAB_SIZE}, d_model={D_MODEL}, "
    f"n_heads={N_HEADS}, d_ff={D_FF}, n_layers={N_LAYERS}"
)
print(f"Total parameters: {n_params:,}")
print()
src_t = torch.tensor([[3, 1, 4, 1]])
tgt_t = torch.tensor([[BOS, 1, 4, 1]])
logits, ca_w = model(src_t, tgt_t)
print(f"Forward pass: src {tuple(src_t.shape)}  tgt_in {tuple(tgt_t.shape)}")
print(f"  -> logits {tuple(logits.shape)}   (B, T, vocab_size)")
print(f"  -> ca_w   {tuple(ca_w.shape)}  (B, n_heads, T, S)")

### Code Walkthrough: Full Encoder-Decoder Architecture

**What just ran — 4 key patterns:**

---

**`DecoderBlock.forward` — three sub-layers in strict order**

The three operations must run in this sequence: (1) causal self-attention on the decoder's own past, (2) cross-attention querying the encoder, (3) feed-forward. The ordering matters: self-attention first lets the decoder incorporate its own generated context before querying the encoder, so the cross-attention Q already "knows what has been generated so far."

---

**`EncoderDecoder.encode(src_ids)` — reuses `MiniEncoder`, run once, reused by every decoder block**

`self.encoder(src_ids)` calls the already-defined, already-sanity-checked `MiniEncoder` and returns `(B, S, d_model)` — one enriched vector per source token. This tensor is passed as `encoder_out` to every `DecoderBlock` inside `decode()`. Because the encoder runs only **once** per source sequence, generating 100 output tokens costs just 1 encoder pass plus 100 decoder passes. Re-encoding at every step would be 100× more expensive.

---

**`causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)` — future tokens blocked**

`diagonal=1` leaves `(i, i)` unmasked — each position can attend to itself. Only positions `j > i` (future tokens) are masked with `-inf` before softmax. Without this mask, the decoder could trivially "cheat" during training by reading the answer token directly from the shifted target sequence.

---

**`self.lm_head = nn.Linear(d_model, vocab_size, bias=False)` — projects hidden state to vocabulary**

The final linear layer maps each decoder position's `d_model`-dimensional vector to `vocab_size` unnormalised logits. Cross-entropy loss is computed against target token IDs. The `bias=False` convention (standard in large transformers) is consistent with weight tying between the input embedding and the output projection — this notebook does not tie those weights (see the closing ledger), but keeps the convention for consistency with production code.

> **PyTorch shape note:** `logits` is `(B, T, vocab_size)`. `.view(-1, vocab_size)` flattens batch and time dims for `nn.CrossEntropyLoss`, giving `(B*T, vocab_size)` predictions against `(B*T,)` targets — loss is averaged over all generated positions.


In [ ]:
# ── Training loop: teacher-forced seq2seq on reversal task ────────────────────
#
# Teacher forcing:
#   decoder input  = [BOS] + target[:-1]
#   decoder target = target + [EOS]
# Loss: cross-entropy over all target positions (including EOS).

import torch.utils.data as data_utils


def build_dataset(srcs, tgts):
    """Pack lists of int sequences into TensorDataset."""
    src_t = torch.tensor(srcs, dtype=torch.long)
    tgt_in = torch.cat(
        [
            torch.full((len(tgts), 1), BOS, dtype=torch.long),
            torch.tensor(tgts, dtype=torch.long),
        ],
        dim=1,
    )
    tgt_out = torch.cat(
        [
            torch.tensor(tgts, dtype=torch.long),
            torch.full((len(tgts), 1), EOS, dtype=torch.long),
        ],
        dim=1,
    )
    return data_utils.TensorDataset(src_t, tgt_in, tgt_out)


train_ds = build_dataset(train_srcs, train_tgts)
val_ds = build_dataset(val_srcs, val_tgts)
train_loader = data_utils.DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = data_utils.DataLoader(val_ds, batch_size=64, shuffle=False)

torch.manual_seed(42)
model = EncoderDecoder(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

EPOCHS = 30
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for src, tgt_in, tgt_out in train_loader:
        optimizer.zero_grad()
        logits, _ = model(src, tgt_in)
        loss = criterion(logits.view(-1, VOCAB_SIZE), tgt_out.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    ep_loss /= len(train_loader)
    train_losses.append(ep_loss)

    model.eval()
    with torch.no_grad():
        v_loss = sum(
            criterion(model(s, ti)[0].view(-1, VOCAB_SIZE), to.view(-1)).item()
            for s, ti, to in val_loader
        ) / len(val_loader)
    val_losses.append(v_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  train={ep_loss:.4f}  val={v_loss:.4f}")

print()
print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final val   loss : {val_losses[-1]:.4f}")
print("  -> Random baseline loss: ~2.56 (log(13), uniform over 13 tokens)")

### Code Walkthrough: Teacher-Forced Seq2Seq Training Loop

**What just ran — 4 key patterns:**

---

**`build_dataset` — teacher forcing: decoder input is the target shifted right**
The decoder receives `[BOS] + target[:-1]` as input and is asked to predict `target + [EOS]`. At training step `t`, the decoder sees the _ground-truth_ token `t−1`, not its own previous prediction. This "teacher forcing" makes gradients clean and convergence fast — but creates a training/inference mismatch: at inference the decoder must use its own (possibly wrong) outputs.

---

**`nn.CrossEntropyLoss(ignore_index=PAD)` — padding positions contribute zero gradient**
Sequences in a batch are padded to the same length. Without `ignore_index=PAD`, the model would waste gradient steps learning to predict padding tokens (which carry no meaning). With it, padded positions are silently excluded from the loss — only real token positions drive learning.

---

**`clip_grad_norm_(model.parameters(), 1.0)` — prevents exploding gradients**
Transformer training occasionally produces very large gradient norms when a loss spike occurs. Clipping the entire gradient vector to unit norm ensures no single noisy batch can undo hundreds of stable update steps. The threshold `1.0` is the standard default for transformer-scale training.

---

**`model.eval()` + `torch.no_grad()` in the validation block — two distinct switches**
`model.eval()` disables dropout and switches batch-norm to running statistics. `torch.no_grad()` stops PyTorch building a computation graph, saving ~40% of memory. Omitting either causes silent bugs: `model.train()` mode gives valid loss numbers but wastes memory and can change behaviour if dropout layers are present.


#### Predict before you run — what accuracy will the trained model achieve?

We just trained for 30 epochs on 2,000 reversal examples with `d_model=32`.

**Predict:** The validation sequence accuracy (all 4 digits must be correct to count) will be:

A. Below 50% — the model barely learns
B. 50-85% — partial learning, many errors
C. Above 90% — the model has cracked the reversal pattern

Write your answer, then run the loss-curve-and-accuracy cell below to measure it.


In [ ]:
# ── Training loss curve + validation sequence accuracy ────────────────────────

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, "b-o", ms=3, label="Train loss")
ax.plot(range(1, EPOCHS + 1), val_losses, "r-o", ms=3, label="Val   loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("EncoderDecoder training curve — sequence reversal task")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Sequence-level accuracy: all SEQ_LEN tokens must be correct
model.eval()
correct = total = 0
with torch.no_grad():
    for src, tgt_in, tgt_out in val_loader:
        logits, _ = model(src, tgt_in)
        preds = logits.argmax(dim=-1)  # (B, T)
        match = (preds[:, :SEQ_LEN] == tgt_out[:, :SEQ_LEN]).all(dim=1)
        correct += match.sum().item()
        total += src.shape[0]

accuracy = correct / total
print(
    f"Validation sequence accuracy: {accuracy:.1%}  ({correct}/{total} fully correct)"
)
print()
if accuracy > 0.90:
    print("  -> Excellent: the model has learned the reversal pattern.")
elif accuracy > 0.70:
    print("  -> Good: most sequences correct; a few more epochs would help.")
else:
    print("  -> Still converging — try more epochs or a larger model.")
print()
print("  -> Random baseline: (1/10)^4 = 0.01% (guessing each digit independently)")

#### What just happened — and what is missing

The model trained, loss fell, and validation accuracy is high.
But we have not verified _how_ it solved the task. Did it actually route
decoder step $i$ to source position $S-1-i$ through cross-attention?

That is exactly what the cross-attention heatmap in Part 6 will prove.


---

## Part 6 — The Cross-Attention Map

### Proving the decoder learned reversal through attention routing

For perfect reversal of `[3, 1, 4, 1]` to `[1, 4, 1, 3]`:

| Decoder step | Must output | Source token to attend to | Source position |
| ------------ | ----------- | ------------------------- | --------------- |
| 0            | 1           | 1 (last)                  | 3               |
| 1            | 4           | 4                         | 2               |
| 2            | 1           | 1                         | 1               |
| 3            | 3           | 3 (first)                 | 0               |

If the model learned this, the cross-attention map should be the **anti-diagonal**.
That would be proof that the architecture, not memorisation, solved the task.


In [ ]:
# ── Cross-attention heatmap: does decoder step i attend to source position S-1-i? ──
#
# We extract cross-attention weights from the last decoder block for
# the single example [3, 1, 4, 1] -> [1, 4, 1, 3].

model.eval()
ex_src = torch.tensor([[3, 1, 4, 1]])
ex_tgt_in = torch.tensor([[BOS, 1, 4, 1]])  # teacher-forced

with torch.no_grad():
    logits_ex, ca_w_ex = model(ex_src, ex_tgt_in)

# ca_w_ex: (1, n_heads, T, S)  where T = SEQ_LEN+1 (BOS + 4 targets)
ca_avg = ca_w_ex[0].mean(dim=0).detach().numpy()  # (T, S) avg over heads
ca_display = ca_avg[:SEQ_LEN, :]  # rows 0..3 (generating steps)

src_labels = ["3", "1", "4", "1"]
tgt_labels = ["->1 (step 0)", "->4 (step 1)", "->1 (step 2)", "->3 (step 3)"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: averaged over all heads
ax = axes[0]
sns.heatmap(
    ca_display,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    xticklabels=src_labels,
    yticklabels=tgt_labels,
    linewidths=0.5,
    vmin=0,
    vmax=1,
)
ax.set_title(
    "Cross-attention (avg all heads)\nDecoder step vs. Source position", fontsize=10
)
ax.set_xlabel("Source position (key)")
ax.set_ylabel("Decoder step (query)")

# Right: head 0 only
ax2 = axes[1]
ca_h0 = ca_w_ex[0, 0, :SEQ_LEN, :].detach().numpy()
sns.heatmap(
    ca_h0,
    ax=ax2,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=src_labels,
    yticklabels=tgt_labels,
    linewidths=0.5,
    vmin=0,
    vmax=1,
)
ax2.set_title("Cross-attention (head 0 only)", fontsize=10)
ax2.set_xlabel("Source position (key)")
ax2.set_ylabel("Decoder step (query)")

plt.suptitle(
    f"Cross-attention map: [3,1,4,1] -> [1,4,1,3]", fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Quantify anti-diagonal alignment
anti_diag = sum(ca_display[i, SEQ_LEN - 1 - i] for i in range(SEQ_LEN)) / SEQ_LEN
print(f"Mean attention weight on anti-diagonal positions: {anti_diag:.3f}")
print()
if anti_diag > 0.5:
    print("  -> Strong anti-diagonal pattern confirmed!")
    print("     Decoder step i attends most to source position S-1-i.")
    print("     Answer to the Part 4 prediction: B (anti-diagonal).")
else:
    print("  -> Attention is more diffuse. Try training for more epochs.")
    print("     The pattern should emerge with sufficient convergence.")
print()
preds = logits_ex.argmax(dim=-1)[0, :SEQ_LEN].tolist()
gold = [1, 4, 1, 3]
print(f"Model prediction (greedy): {preds}")
print(f"Gold target               : {gold}")
correct_str = "Correct!" if preds == gold else "Incorrect — try more training epochs."
print(f"  -> {correct_str}")

In [ ]:
# ── FuncAnimation: cross-attention anti-diagonal assembles step by step ─────
# Each frame reveals one more decoder step row in the heatmap.
# Grey cells = steps not yet generated; coloured cells = steps already decided.
# Watch how the diagonal spotlight moves from bottom-right to top-left,
# proving the decoder queries a different source position at each generation step.

from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch
from IPython.display import HTML, display
import matplotlib.cm as mplcm

n_tgt = ca_display.shape[0]  # T = SEQ_LEN decoder steps
n_src = ca_display.shape[1]  # S = source positions

fig_anim, ax_anim = plt.subplots(figsize=(6, 4.5))

# Static legend handles for the categorical grey/coloured distinction below.
# ax_anim.clear() runs every frame, so the legend is re-added inside
# update_ca_anim rather than once outside it — the handles themselves never
# change, only their re-draw is repeated per frame (Section 10.2: an animated
# categorical color needs its own legend, not just a title string).
legend_handles = [
    Patch(facecolor="#b30000", edgecolor="black", label="Revealed step (generated)"),
    Patch(facecolor="#d3d3d3", edgecolor="black", label="Not yet generated (pending)"),
]


def update_ca_anim(frame):
    ax_anim.clear()
    # Reveal rows 0..frame; grey out the rest
    data = np.full_like(ca_display, np.nan)
    data[: frame + 1, :] = ca_display[: frame + 1, :]
    cmap_anim = mplcm.get_cmap("YlOrRd").copy()
    cmap_anim.set_bad(color="#d3d3d3")
    # Only annotate on the final frame so numbers don't flicker
    annot = frame == n_tgt - 1
    y_labels = tgt_labels[: frame + 1] + ["(pending)"] * (n_tgt - frame - 1)
    sns.heatmap(
        data,
        ax=ax_anim,
        annot=annot,
        fmt=".2f",
        cmap=cmap_anim,
        xticklabels=src_labels,
        yticklabels=y_labels,
        linewidths=0.5,
        vmin=0,
        vmax=1,
        cbar=False,
    )
    ax_anim.set_title(
        f"Cross-attention: step {frame} of {n_tgt - 1} revealed\n"
        "(grey = token not yet generated)",
        fontsize=9,
    )
    ax_anim.set_xlabel("Source position (key)")
    ax_anim.set_ylabel("Decoder step (query)")
    ax_anim.tick_params(axis="y", labelsize=8)
    ax_anim.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.28),
        ncol=1,
        fontsize=7,
        frameon=True,
    )


anim = FuncAnimation(
    fig_anim,
    update_ca_anim,
    frames=n_tgt,
    interval=900,
    blit=False,
    repeat=True,
    repeat_delay=1800,
)
plt.close(fig_anim)

print("Watch the anti-diagonal attention pattern assemble itself:")
print("  Frame 0  (generate '1')  → attention peaks at src position 3  (last token)")
print("  Frame 1  (generate '4')  → attention shifts left to src position 2")
print("  Frame 2  (generate '1')  → attention at src position 1")
print("  Frame 3  (generate '3')  → attention at src position 0  (first token)")
print(
    "  The spotlight walks backward across the source — cross-attention learned reversal."
)
display(HTML(anim.to_jshtml(fps=1)))

#### What just happened — and what is missing

The heatmap is the proof: cross-attention learned to route decoder step $i$ to
source position $S-1-i$, exactly the pattern required for reversal. The architecture
did not need to be told this — it emerged from the prediction loss.

But look closely at how that proof was generated: `ex_tgt_in` above was the **ground-truth**
target, not the model's own predictions. Before bridging to real models, one more question needs
answering honestly: did the decoder actually _generate_ anything, or did every check so far quietly
lean on the answer key? The next subsection answers that directly.


---

### 6a. From Teacher-Forced Proof to Free-Running Generation

Every check so far — the validation accuracy in Part 5, the cross-attention heatmap above — fed the
**ground-truth** target into the decoder as `tgt_in` (teacher forcing). A common shorthand for what
we just did is _"the model generates the reversed sequence."_ Taken literally, that shorthand
implies the decoder produced `[1, 4, 1, 3]` token-by-token from its **own** previous guesses. That
is not what happened: at every one of the four steps above, the decoder was handed the correct
previous token from `tgt_in`, not its own prediction. If step 1 had guessed wrong, step 2 would
still have seen the correct token `1` as its input — errors cannot compound under teacher forcing.

Genuine autoregressive generation must feed each predicted token back in as the next step's input,
starting from nothing but `[BOS]`. This is also where a real training/inference mismatch — often
called **exposure bias** — lives: the decoder trains conditioned on gold history, but at real
inference time it only ever sees its own (possibly wrong) history.

**Predict before you run:** will free-running (self-fed) sequence accuracy on the validation set
be higher than, equal to, or lower than the teacher-forced accuracy measured in Part 5?


### Training Inputs and Free-Running Generation

![Teacher forcing during training compared with free-running autoregressive generation at inference](images/teacher-forcing-vs-generation.png)

During teacher forcing, the decoder receives the ground-truth previous target token while training against the next one. During inference, it feeds back its own generated tokens, which is why an early mistake can influence later steps.


In [ ]:
# ── Greedy autoregressive decoding: feed each prediction back in as the next input ──
#
# Unlike every evaluation above, this loop NEVER sees the ground-truth target.
# It starts from [BOS] alone and grows the decoder input one predicted token at a time —
# real inference behaviour for T5/BART/any encoder-decoder model.

def greedy_decode(model, src_ids, max_len=SEQ_LEN):
    '''Autoregressive greedy decoding. Returns (B, max_len) predicted token ids.

    Parameters
    ----------
    model   : trained EncoderDecoder
    src_ids : (B, S) source token ids
    max_len : number of tokens to generate (reversal's output length always equals SEQ_LEN)
    '''
    model.eval()
    B = src_ids.shape[0]
    with torch.no_grad():
        encoder_out = model.encode(src_ids)
        dec_in = torch.full((B, 1), BOS, dtype=torch.long)   # start token only — no gold history
        for _ in range(max_len):
            T = dec_in.shape[1]
            causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
            logits, _ = model.decode(dec_in, encoder_out, causal_mask)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # greedy: highest-prob token
            dec_in = torch.cat([dec_in, next_token], dim=1)             # feed prediction back in
    return dec_in[:, 1:]   # drop the leading BOS


# Free-running accuracy over the full validation set (compare to the teacher-forced number above)
model.eval()
correct_free = total_free = 0
with torch.no_grad():
    for src, _, tgt_out in val_loader:
        preds_free = greedy_decode(model, src, max_len=SEQ_LEN)
        match = (preds_free == tgt_out[:, :SEQ_LEN]).all(dim=1)
        correct_free += match.sum().item()
        total_free   += src.shape[0]

free_accuracy = correct_free / total_free
print(f"Teacher-forced sequence accuracy (Part 5) : {accuracy:.1%}")
print(f"Free-running (greedy) sequence accuracy   : {free_accuracy:.1%}")
print()
if free_accuracy < accuracy:
    print("  -> Free-running accuracy is LOWER: one early wrong token compounds into every later")
    print("     step, because there is no gold history to fall back on. This is exposure bias,")
    print("     measured directly rather than just asserted.")
else:
    print("  -> Free-running accuracy matched (or exceeded) teacher-forced accuracy here — the")
    print("     model's own greedy predictions were reliable enough not to compound errors on")
    print("     this simple a task; exposure bias grows more visible on longer/harder sequences.")


#### Aside: how would beam search differ?

**This is NOT `model.generate(num_beams=k)`.** Greedy decoding commits to the single best token per
step and can't backtrack. Beam search keeps the top-_k_ partial sequences alive instead of one.


In [ ]:
# ── Illustrative beam search: track top-k candidate sequences, not just 1 ────
#
# Disclaimer: no length normalisation, no batching across beams, no early-stop on EOS —
# a from-scratch illustration of the core "keep top-k partial sequences" idea, not a
# production decoder. Reuses model.encode()/model.decode() from greedy_decode above.


def beam_search_decode(model, src_ids, beam_width=3, max_len=SEQ_LEN):
    """Illustrative beam search. Returns (best_tokens, best_log_prob) for the top-scoring sequence."""
    model.eval()
    assert src_ids.shape[0] == 1, "illustrative version: batch size 1 only"
    with torch.no_grad():
        encoder_out = model.encode(src_ids)
        # Each beam: (token_id_list, cumulative_log_prob)
        beams = [([BOS], 0.0)]
        for _ in range(max_len):
            candidates = []
            for tokens, log_prob in beams:
                dec_in = torch.tensor([tokens], dtype=torch.long)
                T = dec_in.shape[1]
                causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
                logits, _ = model.decode(dec_in, encoder_out, causal_mask)
                log_probs = F.log_softmax(logits[0, -1, :], dim=-1)
                topk_logp, topk_idx = log_probs.topk(beam_width)
                for lp, idx in zip(topk_logp.tolist(), topk_idx.tolist()):
                    candidates.append((tokens + [idx], log_prob + lp))
            # Keep only the top beam_width candidates across ALL beams' expansions
            candidates.sort(key=lambda c: c[1], reverse=True)
            beams = candidates[:beam_width]
    best_tokens, best_log_prob = beams[0]
    return best_tokens[1:], best_log_prob  # drop leading BOS


example_src = torch.tensor([[3, 1, 4, 1]])
greedy_tokens = greedy_decode(model, example_src, max_len=SEQ_LEN)[0].tolist()
beam_tokens, beam_logp = beam_search_decode(
    model, example_src, beam_width=3, max_len=SEQ_LEN
)

print(f"Example [3, 1, 4, 1]  (gold reversed = [1, 4, 1, 3])")
print(f"  Greedy decode      : {greedy_tokens}")
print(f"  Beam search (k=3)  : {beam_tokens}   (cumulative log-prob = {beam_logp:.3f})")
print()
outcome = (
    "matched greedy"
    if greedy_tokens == beam_tokens
    else "found a different, higher-scoring sequence"
)
print(
    f"  -> Beam search {outcome} — it pays off most when the single greedy path isn't globally best."
)
print()
print(
    "Production alternative: HuggingFace's model.generate(num_beams=3, ...) — adds length"
)
print(
    "normalisation, batched beam execution, and per-beam EOS handling on top of this idea."
)

#### What just happened — and what is missing

Free-running generation and beam search both reuse the identical encoder output and decoder
blocks — generation is an _inference-time_ choice, not a different model. What remains is bridging
this toy architecture to the production model families it mirrors. Part 7 does exactly that.


---

## Part 7 — Toy to Real: T5 / BART

### Parameter mapping table

Every hyperparameter in our toy model has a direct counterpart in production
encoder-decoder models. The architecture is identical; only the scale changes.

| Hyperparameter         | Toy (this notebook)  | T5-small                    | BART-base                     |
| ---------------------- | -------------------- | --------------------------- | ----------------------------- |
| `d_model`              | 32                   | 512                         | 768                           |
| `n_heads`              | 4                    | 8                           | 12                            |
| `d_ff`                 | 64                   | 2,048                       | 3,072                         |
| `n_layers` (each side) | 2                    | 6                           | 6                             |
| `vocab_size`           | 13                   | 32,128                      | 50,265                        |
| Parameters             | ~20 K                | ~60 M                       | ~139 M                        |
| Pre-training task      | Sequence reversal    | Span denoising (C4)         | Denoising (Books + Wikipedia) |
| Key use-case           | Toy proof-of-concept | Summarisation / translation | Summarisation / translation   |

Every class we wrote — `MultiHeadSelfAttention`, `CrossAttention`, `EncoderBlock`,
`DecoderBlock` — is a scaled-up copy of what lives inside T5 and BART.
The cross-attention formula $Q_{\text{dec}}(K_{\text{enc}})^\top / \sqrt{d_k}$ is
word-for-word identical; only the tensor widths differ.

**What "span denoising" and "denoising" actually mean (named above, not built here):** our toy
model is trained on a fully-supervised task — every source sequence has one known-correct target.
T5 and BART are _pre-trained_ instead on self-supervised denoising: a clean passage of real text is
corrupted (T5 masks out contiguous **spans** of tokens and asks the decoder to reconstruct just
those spans; BART additionally deletes, shuffles, or masks whole tokens/sentences and asks the
decoder to reconstruct the **entire** original passage), and the network learns general language
structure from millions of such examples before ever seeing a labelled translation/summarisation
pair. This notebook does not build either objective — reversal's supervised target is the simpler
setting to prove cross-attention itself; span-corruption/denoising is a data-construction technique
layered on top of the identical architecture built above.


### The Same Pattern at Production Scale

![Comparison of toy and production-scale encoder-decoder Transformers, including T5 and BART model families](images/toy-to-t5-bart.png)

Scaling adds depth, heads, representation width, context capacity, and training data while preserving the essential pattern: bidirectional source encoding, cross-attention, and causal target generation. T5 and BART are encoder-decoder model families with different pre-training objectives, not interchangeable implementations.


In [ ]:
# ── T5-small summarisation demo (HuggingFace Transformers) ────────────────────
#
# Guarded by try/except — works offline if t5-small weights are already cached.
# If not, a graceful message explains how to cache them.
# The toy model trained above uses the identical cross-attention mechanism.

try:
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    import warnings

    warnings.filterwarnings("ignore")

    print("Loading T5-small (may download ~240 MB on first run)...")
    tokenizer = T5Tokenizer.from_pretrained("t5-small", legacy=False)
    t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")
    t5_model.eval()

    text = (
        "summarize: The encoder-decoder transformer uses cross-attention to bridge "
        "a source sequence and a target sequence. The encoder reads the full source "
        "bidirectionally and produces a rich context map. The decoder generates output "
        "tokens autoregressively, querying the encoder context at every step. "
        "This architecture underlies T5, BART, and the original Transformer paper."
    )

    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)
    with torch.no_grad():
        out_ids = t5_model.generate(**inputs, max_new_tokens=60)
    summary = tokenizer.decode(out_ids[0], skip_special_tokens=True)

    print("T5-small summarisation:")
    print(f"  Input : {text[12:120]}...")
    print(f"  Output: {summary}")
    print()
    print("  -> T5 uses the exact same cross-attention: Q=decoder, K=V=encoder.")
    print("  -> Differences from our toy: d_model=512, n_heads=8, n_layers=6,")
    print("     vocab=32128, trained on C4 corpus (~750 GB text).")

except Exception as e:
    print(f"T5 demo skipped ({type(e).__name__}: {e})")
    print()
    print("To enable: pip install transformers  then re-run this cell.")
    print("  T5 weights (~240 MB) will be downloaded and cached automatically.")
    print()
    print(
        "The toy model you trained above uses the IDENTICAL cross-attention mechanism."
    )
    print("  Toy  d_model=32    T5-small d_model=512")
    print("  Toy  vocab=13      T5-small vocab=32,128")
    print("  Architecture: identical in every structural detail.")

#### Your turn — change one variable and predict

The model was trained on length-4 sequences with `d_model=32`.

```python
# In the Setup cell, change:
SEQ_LEN = 6    # CHANGE: what happens to the cross-attention map dimensions?
D_MODEL = 64   # CHANGE: more capacity — predict convergence speed?
```

**Predict before running:**

1. For `SEQ_LEN=6`, what is the shape of the cross-attention weight tensor?
2. Does the anti-diagonal pattern still appear for length-6 sequences?
3. Does doubling `D_MODEL` help or hurt training speed? (More capacity vs. more
   parameters to optimise.)

Then retrain and compare the cross-attention heatmap.


---

## What This Notebook Covered (and What It Didn't)

Before the completed roadmap below, here is every technique named anywhere in this notebook, sorted
into the tier its actual treatment earns.

### Tier 1 — Implemented and Demonstrated

- **Bidirectional encoder self-attention** (`mask=None`) — proven against causal attention with a
  shared-weights, same-input heatmap comparison (Part 2).
- **Causal decoder self-attention** (`mask=causal_mask`) — same comparison, opposite mask.
- **Cross-attention** (Q=decoder, K=V=encoder, asymmetric `(T×S)` score matrix) — built as its own
  class, shape-verified with `T != S`, and visualised as a trained attention map (Parts 4 and 6).
- **The pre-attention seq2seq bottleneck** — measured directly via cosine similarity between
  per-position and mean-pooled encoder vectors on two different source sequences (Part 3).
- **Teacher-forced training** on the reversal task, with real loss curves and a measured validation
  sequence accuracy (Part 5).
- **Cross-attention's learned routing** — the anti-diagonal attention pattern is measured (mean
  weight on the anti-diagonal, not asserted) and animated frame-by-frame (Part 6).
- **Free-running (greedy) autoregressive generation** — a real `greedy_decode` loop that never sees
  the gold target, with free-running accuracy measured against the teacher-forced number above and
  the gap explained as exposure bias (Part 6a).
- **Toy → real bridge** — a live `t5-small` summarisation call (guarded by a graceful fallback if
  offline), plus the full toy→production hyperparameter mapping table (Part 7).

### Tier 2 — Explained but Not Fully Implemented

- **Beam search decoding** — a from-scratch, illustrative `beam_search_decode` (top-`k` partial
  sequences by cumulative log-probability) runs on the trained model and is compared to greedy
  output, but it skips length normalisation, batched beam execution, and per-beam EOS handling that
  `model.generate(num_beams=k)` provides in production (Part 6b). Reason: the point here is _why_
  tracking more than one candidate can beat greedy, not shipping a production-grade decoder.
- **T5 span-corruption / BART denoising pre-training objectives** — explained in plain English
  (what each corruption/reconstruction scheme does and why it lets the model learn from unlabelled
  text) immediately after the toy→real parameter table, but not built (Part 7). Reason: this
  notebook's reversal task is fully supervised by design, specifically to keep the focus on
  cross-attention mechanics rather than large-scale self-supervised pre-training.
- **Weight tying between the token embedding and the LM head** — named as the reason `lm_head` uses
  `bias=False` (a convention this notebook follows), but the embedding and `lm_head` weights are
  never actually tied in code (Part 5). Reason: tying is a parameter-saving optimisation aimed at
  production-scale vocabularies; at `vocab_size=13` the savings are negligible.
- **Padding masks for variable-length batches** — `nn.CrossEntropyLoss(ignore_index=PAD)` is real,
  running code, but every source/target sequence in this notebook is a fixed `SEQ_LEN=4`, so no
  batch actually contains a padded position — the mechanism has never been exercised on real
  variable-length input (Part 5). Reason: reversal's fixed length keeps every visualisation the same
  shape end to end; a real translation task would need this exercised for real.

### Tier 3 — Named but Out of Scope

- **KV-caching across generation steps** — the production technique that avoids recomputing past
  decoder keys/values at every generation step. Out of scope: `greedy_decode` above already gets its
  speed from encoding once; caching decoder K/V across steps is a real optimisation this notebook
  doesn't need at `SEQ_LEN=4` to make its point.
- **Pre-attention RNN encoder-decoders and Bahdanau/Luong attention** — the historical bridge
  between the fixed-vector bottleneck (Part 3) and modern cross-attention. Out of scope: covering
  the actual RNN recurrence and the original additive/multiplicative attention formulas would double
  this notebook's scope for a point that's about the _bottleneck_, which is already measured
  directly.
- **Multilingual encoder-decoders (mT5, mBART, NLLB)** — same architecture as T5/BART, scaled to
  100+ languages with a shared multilingual vocabulary. Out of scope: no new architectural idea over
  what Part 7 already bridges to.
- **Machine translation and question-answering as applications** — encoder-decoder's other two
  headline use-cases beyond the summarisation demoed live in Part 7. Out of scope: the underlying
  mechanism (cross-attention over an encoded source) is identical across all three; only the
  training data changes.
- **Why decoder-only architectures (GPT-3/4, LLaMA) came to dominate general-purpose LLMs despite
  encoder-decoder's structural fit for seq2seq tasks** — a real, actively-discussed design-space
  question (compute/deployment trade-offs, one model serving every task). Out of scope: it's a
  systems/product question that sits a level above this notebook's architecture-mechanics focus.


---

## Summary — Completed Roadmap

| Step | Part | Concept | Key Idea |
|------|------|---------|----------|
| 1 | The Contract | What encoder-decoder solves | Bidirectionality + variable-length I/O |
| 2 | The Encoder | Bidirectional attention | mask=None gives every token a 360-degree view |
| 3 | The Bottleneck | Why naive pooling fails | Fixed vector loses positional detail at scale |
| 4 | Cross-Attention | The bridge | Q = decoder, K = V = encoder; asymmetric $(T \times S)$ scores |
| 5 | Full Model + Training | Wire all components | Teacher-forced seq2seq converges on reversal |
| 6 | Cross-Attention Map | Visualise learned routing | Anti-diagonal confirms decoder step $i$ -> source $S-1-i$ |
| 6a | Free-Running Decoding | Free-running greedy decode (+ beam-search aside) | Teacher-forced != free-running; exposure bias measured, not assumed |
| 7 | Toy to Real | T5 / BART mapping | Same architecture, wider vectors, larger vocab |

---

### Key insights to keep

- The **only** code difference between an encoder block and a decoder block is
  `mask=None` vs `mask=causal_mask` — one argument controls bidirectionality.
- Cross-attention score matrix shape is $(T_{\text{tgt}} \times S_{\text{src}})$ —
  **asymmetric**, unlike self-attention's $(S \times S)$. No mask on the encoder side.
- The information bottleneck in pre-attention seq2seq came from collapsing the source
  into a fixed vector. Cross-attention replaces it with $S$ live source vectors — the
  capacity grows linearly with source length.
- The anti-diagonal heatmap is not an artifact — it is **proof** that the architecture
  solved the reversal task through learned attention routing, not through memorisation.
- T5-small has 60 M parameters and $d_{\text{model}}=512$; our toy has ~20 K and
  $d_{\text{model}}=32$. The cross-attention formula $Q(K^\top)/\sqrt{d_k}$ is
  unchanged — just wider vectors.


---

## What's Next — and Why Decoder-Only

This notebook built the full encoder-decoder architecture used by T5 and BART. The next chapter (`04-llm/`) takes a different path: **decoder-only** models like GPT-2.

| Architecture                | Strength                                                                                     | Weakness at scale                                                                                             |
| --------------------------- | -------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------- |
| Encoder-Decoder (T5, BART)  | Optimal for fixed-format transductions (translation, summarisation, Q→A with known length)   | Two separate stacks to store and serve; encoder parameters add cost without benefit for open-ended generation |
| Decoder-Only (GPT-2, LLaMA) | Single stack; autoregressive pretraining on raw text scales to arbitrary tasks via prompting | No explicit bidirectional context over the source — must include source in the prompt                         |
| Encoder-Only (BERT)         | Best for classification, retrieval, embedding                                                | Cannot generate                                                                                               |

At the scale of GPT-3 (175B parameters), training an encoder-decoder model would have required double the parameter budget for tasks where the encoder's bidirectional attention didn't pay off. Decoder-only models generalised surprisingly well to translation and summarisation via few-shot prompting — removing the case for the extra encoder stack.

→ **Next:** `04-llm/01-llm-finetuning-data-techniques.ipynb` — fine-tuning a decoder-only model on a private corpus under real compute constraints.
